# DK vs IT — Intangible Capital & Labour Productivity Divergence

This notebook investigates the divergence in labour productivity (LP) growth between **Denmark** and **Italy** from 1996 to 2019, with a focus on the role of **intangible capital investment**. The central research question is: *to what extent do differences in intangible capital accumulation — organisational capital, software & databases, and ICT tangible assets — explain the widening LP gap between the two countries, particularly in the ICT sector (J) and in the total economy (TOT_IND)?* We decompose the LP growth gap into contributions from intangible capital, TFP, ICT tangible capital, and a residual, drawing on industry-level data from the FINAL_DK_IT dataset.

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path


# ── Load data ──────────────────────────────────────────────────────────────────
HERE     = os.path.abspath(os.path.dirname('')) if '__file__' not in dir() else os.path.dirname(__file__)
DATA_DIR = os.path.join(HERE, 'data')
PATH     = os.path.join(DATA_DIR, 'FINAL_DK_IT.csv')

df = pd.read_csv(PATH)


# ── Colour palette ─────────────────────────────────────────────────────────────
DK_COLOR   = "#1B4F8A"
IT_COLOR   = "#C0392B"
INTANG_C   = "#2980B9"
TFP_C      = "#E67E22"
ICT_C      = "#27AE60"
OTHER_C    = "#BDC3C7"

# ── Matplotlib style ───────────────────────────────────────────────────────────
plt.rcParams.update({
    "font.family":        "serif",
    "font.size":          11,
    "axes.spines.top":    False,
    "axes.spines.right":  False,
    "axes.grid":          True,
    "grid.color":         "#e0e0e0",
    "grid.linestyle":     "--",
    "grid.linewidth":     0.6,
    "axes.titlesize":     12,
    "axes.labelsize":     10,
    "legend.fontsize":    9,
    "legend.framealpha":  0.7,
})

# ── Helper ─────────────────────────────────────────────────────────────────────
def get_country_sector(country: str, sector: str) -> pd.DataFrame:
    """Filter df by country and industry, return indexed by Year."""
    mask = (df["Country"] == country) & (df["Industry"] == sector)
    return df.loc[mask].set_index("Year").sort_index()

print("Data loaded:", df.shape)
print("Countries:",  df["Country"].unique())
print("Industries:", df["Industry"].unique())
print("Years:",      df["Year"].min(), "–", df["Year"].max())
df.head()

ParserError: Error tokenizing data. C error: Expected 1 fields in line 50, saw 2


## Chart 1 — Labour Productivity Growth: DK vs IT, ICT Sector (J)

In [ ]:
dk_j = get_country_sector("Denmark", "J")
it_j = get_country_sector("Italy",   "J")

years = dk_j.index

fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(years, dk_j["LP1_G"], color=DK_COLOR, lw=2,   marker="o", markersize=4, label="Denmark")
ax.plot(years, it_j["LP1_G"], color=IT_COLOR, lw=2,   marker="s", markersize=4, label="Italy")

ax.fill_between(
    years,
    dk_j["LP1_G"], it_j["LP1_G"],
    where=(dk_j["LP1_G"] >= it_j["LP1_G"]),
    alpha=0.15, color=DK_COLOR, label="DK above IT"
)
ax.fill_between(
    years,
    dk_j["LP1_G"], it_j["LP1_G"],
    where=(dk_j["LP1_G"] < it_j["LP1_G"]),
    alpha=0.15, color=IT_COLOR, label="IT above DK"
)

ax.axhline(0, color="black", lw=0.8, linestyle="-")

ax.set_title("Labour Productivity Growth — ICT Sector (J) — Denmark vs Italy, 1996–2019")
ax.set_xlabel("Year")
ax.set_ylabel("LP Growth (index)")
ax.legend()
fig.tight_layout()

fig.savefig("chart1_lp1g_divergence.png", dpi=150)
plt.show()

## Chart 2 — Decomposition of the LP Growth Gap, ICT Sector (J)

In [ ]:
dk_j = get_country_sector("Denmark", "J")
it_j = get_country_sector("Italy",   "J")

years = dk_j.index

gap_LP1G   = dk_j["LP1_G"]          - it_j["LP1_G"]
gap_Intang = dk_j["LP1ConIntang"]    - it_j["LP1ConIntang"]
gap_TFP    = dk_j["LP1ConTFP"]       - it_j["LP1ConTFP"]
gap_ICT    = dk_j["LP1ConTangICT"]   - it_j["LP1ConTangICT"]
gap_other  = gap_LP1G - gap_Intang - gap_TFP - gap_ICT

components = {
    "Intangible capital": (gap_Intang, INTANG_C),
    "TFP":                (gap_TFP,    TFP_C),
    "ICT tangible":       (gap_ICT,    ICT_C),
    "Other / residual":   (gap_other,  OTHER_C),
}

fig, ax = plt.subplots(figsize=(10, 5))

pos_bottom = np.zeros(len(years))
neg_bottom = np.zeros(len(years))

for label, (series, color) in components.items():
    vals = series.values.astype(float)
    pos_vals = np.where(vals > 0, vals, 0)
    neg_vals = np.where(vals < 0, vals, 0)
    ax.bar(years, pos_vals, bottom=pos_bottom, color=color, label=label, width=0.7, alpha=0.85)
    ax.bar(years, neg_vals, bottom=neg_bottom, color=color,              width=0.7, alpha=0.85)
    pos_bottom += pos_vals
    neg_bottom += neg_vals

ax.plot(years, gap_LP1G, color="black", lw=1.5, marker="D", markersize=5,
        zorder=5, label="Total gap (DK − IT)")
ax.axhline(0, color="black", lw=0.8)

ax.set_title("Decomposition of LP Growth Gap (DK − IT) — ICT Sector (J), 1996–2019")
ax.set_xlabel("Year")
ax.set_ylabel("Gap contribution (index points)")
ax.legend(ncol=2)
fig.tight_layout()

fig.savefig("chart2_gap_decomposition.png", dpi=150)
plt.show()

## Chart 3 — Contribution of Intangible Capital: DK vs IT, ICT Sector (J)

In [ ]:
dk_j = get_country_sector("Denmark", "J")
it_j = get_country_sector("Italy",   "J")

years = dk_j.index

fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(years, dk_j["LP1ConIntang"], color=DK_COLOR, lw=2, marker="o", markersize=4, label="Denmark")
ax.plot(years, it_j["LP1ConIntang"], color=IT_COLOR, lw=2, marker="s", markersize=4, label="Italy")

ax.fill_between(
    years,
    dk_j["LP1ConIntang"], it_j["LP1ConIntang"],
    where=(dk_j["LP1ConIntang"] >= it_j["LP1ConIntang"]),
    alpha=0.15, color=DK_COLOR
)
ax.fill_between(
    years,
    dk_j["LP1ConIntang"], it_j["LP1ConIntang"],
    where=(dk_j["LP1ConIntang"] < it_j["LP1ConIntang"]),
    alpha=0.15, color=IT_COLOR
)

ax.axhline(0, color="black", lw=0.8)
ax.set_title("Contribution of Intangible Capital to LP Growth — ICT Sector (J), 1996–2019")
ax.set_xlabel("Year")
ax.set_ylabel("LP contribution of intangible capital (index)")
ax.legend()
fig.tight_layout()

fig.savefig("chart3_intang_contribution.png", dpi=150)
plt.show()

## Chart 4 — Intangible Investment Composition: DK vs IT, ICT Sector (J)

In [ ]:
dk_j = get_country_sector("Denmark", "J")
it_j = get_country_sector("Italy",   "J")

years = dk_j.index

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# ── Left panel: total intangible investment levels ────────────────────────────
ax1.plot(years, dk_j["I_Intang"], color=DK_COLOR, lw=2, marker="o", markersize=4, label="Denmark")
ax1.plot(years, it_j["I_Intang"], color=IT_COLOR, lw=2, marker="s", markersize=4, label="Italy")
ax1.set_title("Total Intangible Investment (I_Intang)")
ax1.set_xlabel("Year")
ax1.set_ylabel("Investment level")
ax1.legend()

# ── Right panel: OrgCap & Soft_DB as % of I_Intang ───────────────────────────
dk_orgcap_share  = dk_j["I_OrgCap"]   / dk_j["I_Intang"] * 100
dk_soft_share    = dk_j["I_Soft_DB"]  / dk_j["I_Intang"] * 100
it_orgcap_share  = it_j["I_OrgCap"]   / it_j["I_Intang"] * 100
it_soft_share    = it_j["I_Soft_DB"]  / it_j["I_Intang"] * 100

ax2.plot(years, dk_orgcap_share, color=DK_COLOR, lw=2, linestyle="-",  marker="o", markersize=4,
         label="DK — OrgCap share")
ax2.plot(years, it_orgcap_share, color=IT_COLOR, lw=2, linestyle="-",  marker="s", markersize=4,
         label="IT — OrgCap share")
ax2.plot(years, dk_soft_share,   color=DK_COLOR, lw=2, linestyle="--", marker="o", markersize=4,
         label="DK — Soft&DB share")
ax2.plot(years, it_soft_share,   color=IT_COLOR, lw=2, linestyle="--", marker="s", markersize=4,
         label="IT — Soft&DB share")

ax2.yaxis.set_major_formatter(mticker.PercentFormatter())
ax2.set_title("OrgCap & Soft/DB as % of Total Intangible Investment")
ax2.set_xlabel("Year")
ax2.set_ylabel("Share (%)")
ax2.legend()

fig.tight_layout()
fig.savefig("chart4_investment_composition.png", dpi=150)
plt.show()

## Chart 5 — Benchmark: Total Economy (TOT_IND)

In [ ]:
dk_tot = get_country_sector("Denmark", "TOT_IND")
it_tot = get_country_sector("Italy",   "TOT_IND")

years = dk_tot.index

gap_LP1G   = dk_tot["LP1_G"]        - it_tot["LP1_G"]
gap_Intang = dk_tot["LP1ConIntang"]  - it_tot["LP1ConIntang"]
gap_TFP    = dk_tot["LP1ConTFP"]     - it_tot["LP1ConTFP"]
gap_ICT    = dk_tot["LP1ConTangICT"] - it_tot["LP1ConTangICT"]
gap_other  = gap_LP1G - gap_Intang - gap_TFP - gap_ICT

components = {
    "Intangible capital": (gap_Intang, INTANG_C),
    "TFP":                (gap_TFP,    TFP_C),
    "ICT tangible":       (gap_ICT,    ICT_C),
    "Other / residual":   (gap_other,  OTHER_C),
}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# ── Left panel: LP1_G levels ──────────────────────────────────────────────────
ax1.plot(years, dk_tot["LP1_G"], color=DK_COLOR, lw=2, marker="o", markersize=4, label="Denmark")
ax1.plot(years, it_tot["LP1_G"], color=IT_COLOR, lw=2, marker="s", markersize=4, label="Italy")
ax1.fill_between(
    years,
    dk_tot["LP1_G"], it_tot["LP1_G"],
    where=(dk_tot["LP1_G"] >= it_tot["LP1_G"]),
    alpha=0.15, color=DK_COLOR
)
ax1.fill_between(
    years,
    dk_tot["LP1_G"], it_tot["LP1_G"],
    where=(dk_tot["LP1_G"] < it_tot["LP1_G"]),
    alpha=0.15, color=IT_COLOR
)
ax1.axhline(0, color="black", lw=0.8)
ax1.set_title("LP Growth — Total Economy (TOT_IND), 1996–2019")
ax1.set_xlabel("Year")
ax1.set_ylabel("LP Growth (index)")
ax1.legend()

# ── Right panel: gap decomposition stacked bar ────────────────────────────────
pos_bottom = np.zeros(len(years))
neg_bottom = np.zeros(len(years))

for label, (series, color) in components.items():
    vals = series.values.astype(float)
    pos_vals = np.where(vals > 0, vals, 0)
    neg_vals = np.where(vals < 0, vals, 0)
    ax2.bar(years, pos_vals, bottom=pos_bottom, color=color, label=label, width=0.7, alpha=0.85)
    ax2.bar(years, neg_vals, bottom=neg_bottom, color=color,              width=0.7, alpha=0.85)
    pos_bottom += pos_vals
    neg_bottom += neg_vals

ax2.plot(years, gap_LP1G, color="black", lw=1.5, marker="D", markersize=5,
         zorder=5, label="Total gap (DK − IT)")
ax2.axhline(0, color="black", lw=0.8)
ax2.set_title("LP Gap Decomposition — Total Economy (TOT_IND), 1996–2019")
ax2.set_xlabel("Year")
ax2.set_ylabel("Gap contribution (index points)")
ax2.legend(ncol=2)

fig.tight_layout()
fig.savefig("chart5_totind_benchmark.png", dpi=150)
plt.show()

## Summary of Key Findings

- **Chart 1 — LP growth in the ICT sector (J):** Denmark consistently outperforms Italy in labour productivity growth in the ICT sector over the 1996–2019 period, with the gap widening especially after the mid-2000s. Italy shows near-zero or negative LP growth for extended stretches, suggesting structural weaknesses in this high-value sector.

- **Chart 2 — Gap decomposition, ICT sector (J):** The positive LP growth gap in favour of Denmark is driven primarily by higher TFP growth and intangible capital contributions. Italy's underperformance in organisational capital and software & database investment accounts for a sizeable negative share of the gap, especially in the post-2005 years. The residual component is non-trivial, reflecting measurement limits.

- **Chart 3 — Intangible capital contribution, ICT sector (J):** Denmark's intangible capital contribution to LP growth is persistently higher and more stable than Italy's. Italy's contribution fluctuates and often turns negative, indicating that intangible capital accumulation in the Italian ICT sector has been insufficient to sustain productivity gains.

- **Chart 4 — Intangible investment composition, ICT sector (J):** Denmark invests substantially more in intangibles overall. Within the intangible basket, Denmark allocates a larger and growing share to organisational capital, while Italy's intangible mix is more software-intensive but lower in absolute terms. This structural difference in the *composition* of intangible investment may explain differences in how productively investments translate into LP gains.

- **Chart 5 — Total economy benchmark (TOT_IND):** The LP divergence is not confined to the ICT sector: at the total-economy level Denmark also outperforms Italy, though the gap is smaller and more volatile. TFP differences remain the dominant source of divergence, but intangible capital plays a non-trivial supporting role — suggesting that Italy's intangible investment gap is an economy-wide phenomenon, not just a sectoral anomaly.